# Task 2
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [59]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    Matern,
    RationalQuadratic,
    DotProduct,
    WhiteKernel
)


# Data Loading
TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test
(and potentially change initialization of variables to accomodate how you deal with non-numeric data)

In [60]:
"""
This loads the training and test data, preprocesses it, removes the NaN
values and interpolates the missing data using imputation

Parameters
----------
Compute
----------
X_train: matrix of floats, training input with features
y_train: array of floats, training output with labels
X_test: matrix of floats: dim = (100, ?), test input with features
"""
# Load training data
"""train_df = pd.read_csv("train.csv")
    
print("Training data:")
print("Shape:", train_df.shape)
print(train_df.head(2))
print('\n')
    
# Load test data
test_df = pd.read_csv("test.csv")

print("Test data:")
print(test_df.shape)
print(test_df.head(2))

# Dummy initialization of the X_train, X_test and y_train   
# TODO: Depending on how you deal with the non-numeric data, you may want to 
# modify/ignore the initialization of these variables   
X_train = np.zeros_like(train_df.drop(['price_CHF'],axis=1))
y_train = np.zeros_like(train_df['price_CHF'])
X_test = np.zeros_like(test_df)

# TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test

assert (X_train.shape[1] == X_test.shape[1]) and (X_train.shape[0] == y_train.shape[0]) and (X_test.shape[0] == 100), "Invalid data shape" """
"""
This loads the training and test data, preprocesses it, removes rows with missing
target values and prepares the matrices used by the model.

Parameters
----------
Compute
----------
X_train: training input with features
y_train: training output with labels
X_test: test input with features
"""
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Training data shape:", train_df.shape)
print("Test data shape:", test_df.shape)

# Remove rows where the target is missing
train_df = train_df.dropna(subset=["price_CHF"]).reset_index(drop=True)

X_train = train_df.drop(columns=["price_CHF"])
y_train = train_df["price_CHF"].to_numpy(dtype=float)
X_test = test_df.copy()

assert X_train.shape[1] == X_test.shape[1], "Train/test feature mismatch"
assert X_train.shape[0] == y_train.shape[0], "Invalid train shape"

Training data shape: (900, 11)
Test data shape: (100, 10)


# Modeling and Prediction
TODO: Define the model and fit it using training data. Then, use test data to make predictions

In [66]:
"""
This defines the model, fits training data and then does the prediction
with the test data 

Parameters
----------
X_train: matrix of floats, training input with 10 features
y_train: array of floats, training output
X_test: matrix of floats: dim = (100, ?), test input with 10 features

Compute
----------
y_test: array of floats: dim = (100,), predictions on test set
"""
"""class Model(object):
    def __init__(self):
        super().__init__()
        self._x_train = None
        self._y_train = None

    def fit(self, X_train: np.ndarray, y_train: np.ndarray):
        #TODO: Define the model and fit it using (X_train, y_train)
        self._x_train = X_train
        self._y_train = y_train

    def predict(self, X_test: np.ndarray) -> np.ndarray:
        #TODO: Use the model to make predictions y_pred using test data X_test
        y_pred=np.zeros(X_test.shape[0])
        assert y_pred.shape == (X_test.shape[0],), "Invalid data shape"
        return y_pred """
"""
This defines the model, fits training data and predicts on the test data.

Parameters
----------
X_train: training input
y_train: training targets
X_test: test input

Compute
----------
y_test: predictions on test set
"""
from sklearn.impute import KNNImputer


class Model(object):
    def __init__(self):
        super().__init__()
        self.pipeline = None

    def fit(self, X_train: pd.DataFrame, y_train: np.ndarray):
        categorical_features = ["season"]
        numerical_features = [col for col in X_train.columns if col != "season"]

        numeric_transformer = Pipeline(steps=[
            ("imputer", KNNImputer(n_neighbors=7, weights="distance")),
            ("scaler", StandardScaler())
        ])

        categorical_transformer = Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ])

        preprocessor = ColumnTransformer(
            transformers=[
                ("num", numeric_transformer, numerical_features),
                ("cat", categorical_transformer, categorical_features)
            ]
        )

        kernel = (
            ConstantKernel(1.0, (1e-3, 1e3))
            * Matern(length_scale=1.0, nu=1.5)
            + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e0))
        )

        regressor = GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-4,
            normalize_y=True,
            n_restarts_optimizer=3,
            random_state=42
        )

        self.pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("regressor", regressor)
        ])

        self.pipeline.fit(X_train, y_train)

    def predict(self, X_test: pd.DataFrame) -> np.ndarray:
        y_pred = self.pipeline.predict(X_test)
        y_pred = np.asarray(y_pred, dtype=float)

        assert y_pred.shape == (X_test.shape[0],), "Invalid prediction shape"
        return y_pred

In [67]:
model = Model()
# Use this function to fit the model
model.fit(X_train=X_train, y_train=y_train)
# Use this function for inference
y_pred = model.predict(X_test)

# Saving Results
You don't have to change this

In [68]:
dt = pd.DataFrame(y_pred) 
dt.columns = ['price_CHF']
dt.to_csv('results.csv', index=False)
print("\nResults file successfully generated!")


Results file successfully generated!


# Validation 

In [69]:
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_idx, val_idx in kf.split(X_train):
    X_tr = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_tr = y_train[train_idx]
    y_val = y_train[val_idx]

    model = Model()
    model.fit(X_tr, y_tr)
    y_val_pred = model.predict(X_val)

    score = r2_score(y_val, y_val_pred)
    scores.append(score)

print("Fold R^2 scores:", scores)
print("Mean R^2:", np.mean(scores))
print("Std R^2:", np.std(scores))

c:\Users\giaco\anaconda3\envs\IML\Lib\site-packages\sklearn\gaussian_process\_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 15 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


Fold R^2 scores: [0.969872230739373, 0.9710180866358642, 0.9721110169521361, 0.9768473065336034, 0.9587765603129238]
Mean R^2: 0.96972504023478
Std R^2: 0.005966278719041017
